# 4.6 — Network Density and Local Spatial Dependence

How does the availability of nearby stations affect forecast quality?
This notebook presents a controlled local-masking experiment using v27's
MR=0.5 predictions, where 50\% of stations are randomly masked per
evaluation window. By analysing how the fraction of masked local
neighbours correlates with target-station error, we isolate the
contribution of local spatial context.

**Main result**: ΔMAE vs masking radius, with terrain-class breakdown
and a random-removal control that distinguishes genuine local spatial
dependence from simply having fewer stations.

Radii: 5, 10, 25, 50, 100 km.

In [ ]:
import os, sys, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
for _cand in (os.getcwd(),
              os.path.join(os.getcwd(), "notebooks", "analysis"),
              os.path.dirname(os.path.abspath("__file__"))):
    if os.path.isfile(os.path.join(_cand, "common.py")):
        if _cand not in sys.path: sys.path.insert(0, _cand)
        break
import importlib, common as C; importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

RUNS  = C.discovered_runs()
MR0_RUNS = [r for r in RUNS if "mr0.00" in RUNS[r]]
ns = C.norm_stats(); VARS = ns["var_names"]; STD = ns["std"]
stn = C.station_table(); KEEP = C.keep_mask(stn, VARS)
AGG  = {r: C.load_agg(r, "mr0.00") for r in MR0_RUNS}
GRID = AGG[MR0_RUNS[0]]["grid"]; LEAD = C.lead_labels(GRID); K = len(GRID)
NV = len(VARS); N = len(stn)
print("Models at MR=0.00:", MR0_RUNS)

from scipy.stats import pearsonr, spearmanr, linregress
REF_KI = 6  # \u2248 3 h
RADII_KM = [5, 10, 25, 50, 100]  # subset for masking experiment
NN_RADII = [10, 25, 50, 100]  # full set for correlation analysis

TC_IDX = C.terrain_class_indices(stn)
TC_COLORS = {"valley floor": "#2B7A78", "elevated enclosed": "#D4A373",
             "exposed ridge/summit": "#BC4749", "open / slope": "#5B8E7D"}

In [ ]:
# ── Distance matrix from Swiss LV95 coordinates ─────────────────────────────
east  = stn.easting.values.astype(np.float64)
north = stn.northing.values.astype(np.float64)
DIST = np.sqrt((east[:, None] - east[None, :])**2 +
               (north[:, None] - north[None, :])**2) / 1000.0  # km
np.fill_diagonal(DIST, np.inf)  # exclude self-distance

# Adjacency matrices and neighbour counts at masking radii
ADJ = {R: (DIST <= R) for R in RADII_KM}
N_LOCAL = {R: ADJ[R].sum(axis=1) for R in RADII_KM}

# Neighbour counts at all analysis radii
NN_COUNT = {R: (DIST <= R).sum(axis=1) for R in NN_RADII}

print("Neighbour counts at analysis radii:")
for R in NN_RADII:
    n = NN_COUNT[R]
    print(f"  {R:>3d} km: median={np.median(n):.0f}, "
          f"mean={n.mean():.1f}, min={n.min()}, max={n.max()}, "
          f"zero: {(n == 0).sum()}")

# Distance-weighted neighbour availability
# W_i = Σ_j exp(-d_ij / λ), excluding self
DIST_FINITE = DIST.copy()
np.fill_diagonal(DIST_FINITE, 0)  # temp: 0 for self (excluded below)
LAMBDA_KM = 25  # decay scale ≈ median inter-station distance
W_RAW = np.exp(-DIST_FINITE / LAMBDA_KM)
np.fill_diagonal(W_RAW, 0)  # exclude self-contribution
W_i = W_RAW.sum(axis=1)  # (N,)
np.fill_diagonal(DIST_FINITE, np.inf)  # restore

# Nearest-station distance
d_NN = DIST.min(axis=1)  # (N,) — km to nearest station

print(f"W_i (λ={LAMBDA_KM} km): median={np.median(W_i):.2f}, "
      f"min={W_i.min():.2f}, max={W_i.max():.2f}")
print(f"d_NN: median={np.median(d_NN):.1f} km, "
      f"min={d_NN.min():.1f}, max={d_NN.max():.1f} km")


## Local Station Masking Experiment

At MR=0.5, approximately half the stations are randomly masked each
evaluation window. For each (window, target station) pair where the
target is **visible**, we count how many of its neighbours within
radius *R* happened to be masked. Splitting into "high local masking"
(> 50\% of *R*-km neighbours masked) vs "low local masking" gives
ΔMAE — the MAE increase attributable to losing local spatial context.

**Random control**: the same statistic computed with randomly chosen
pseudo-neighbours (same count per station) instead of real spatial
neighbours. If ΔMAE\_local ≫ ΔMAE\_random, the effect is genuinely
spatial, not just a consequence of having fewer total stations.

In [ ]:
# ── Load v27 MR=0.5 dump ────────────────────────────────────────────────────
# At MR=0.5, ~50% of stations are randomly masked per evaluation pass.
# masked_idx[w] lists which stations were masked.
# Predictions exist for ALL stations (decoder reconstructs masked ones).
import torch

d5 = C.load_dump("v27", "mr0.50")
P5 = d5["preds"][:, REF_KI, :, :NV].numpy().astype(np.float64)   # (Mw, N, NV)
T5 = d5["targets"][:, REF_KI, :, :NV].numpy().astype(np.float64)
M5 = d5["masks"][:, REF_KI, :, :NV].numpy() > 0.5                # (Mw, N, NV)
MI5 = d5["masked_idx"].numpy()                                    # (Mw, n_masked)
Mw = P5.shape[0]

# Per-window masked-station boolean
masked_bool = np.zeros((Mw, N), dtype=bool)
for w in range(Mw):
    masked_bool[w, MI5[w]] = True
visible_bool = ~masked_bool  # (Mw, N)

# Per-window absolute error in physical units
err5 = np.abs(P5 - T5) * STD[None]          # (Mw, N, NV)
val5 = M5 & KEEP[None]                      # (Mw, N, NV)

# Local masking counts per (window, station, radius):
# local_masked_count[R][w, i] = number of i's R-km neighbours masked in window w
# Computed as masked_bool @ adjacency_matrix (matrix multiplication).
local_masked_count = {}
for R in RADII_KM:
    adj_f = ADJ[R].astype(np.float32)
    local_masked_count[R] = masked_bool.astype(np.float32) @ adj_f  # (Mw, N)

TH5_full = d5['target_hours'].numpy()  # save before freeing dump
del d5, P5, T5, M5  # free memory
import gc; gc.collect()
print(f"Loaded: {Mw} windows, {MI5.shape[1]} masked/window, {N} stations")

In [ ]:
# ── Load extended aggregators for v27 at MR=0.0 and MR=0.5 ──────────────────
EXT0 = C.load_ext("v27", "mr0.00")
EXT5 = C.load_ext("v27", "mr0.50")
print("Extended aggregators loaded for v27 at MR=0.0 and MR=0.5")


In [ ]:
# ── Per-station visible MAE map at +3 h — MR=0.0 vs MR=0.5 ──────────────────
import matplotlib.colors as mcolors

# The MR=0.5 mask is redrawn every window, so every station has a
# visible-window MAE at MR=0.5 and an all-window MAE at MR=0.0: both panels
# already show the same 155 stations. VIS_IDX = all stations.
VIS_IDX = np.arange(N)

def _station_mae(ext, ki, station_idx=None):
    s = ext["tod_mod_sum"][:, ki, :, :].sum(axis=0)   # (N, V)
    c = ext["tod_mod_cnt"][:, ki, :, :].sum(axis=0)
    mae = np.where(c > 0, s / np.maximum(c, 1), np.nan)
    if station_idx is not None:
        keep = np.zeros(mae.shape[0], dtype=bool)
        keep[station_idx] = True
        mae = np.where(keep[:, None], mae, np.nan)
    return mae

def _station_vis_mae(ext, ki):
    s = ext["tod_mod_vis_sum"][:, ki, :, :].sum(axis=0)  # (N, V)
    c = ext["tod_mod_vis_cnt"][:, ki, :, :].sum(axis=0)
    return np.where(c > 0, s / np.maximum(c, 1), np.nan)

mae0 = _station_mae(EXT0, REF_KI, station_idx=VIS_IDX)  # (N, V) — MR=0.0, same stations
mae5 = _station_vis_mae(EXT5, REF_KI)    # (N, V) — MR=0.5 visible only

fig, axes = plt.subplots(2, NV, figsize=(5.4 * NV, 9), squeeze=False)
labels_mr = ["MR=0.0 (all visible)", "MR=0.5 (visible only)"]

for vi, v in enumerate(VARS):
    # Shared color scale across MR settings
    both = np.concatenate([mae0[:, vi], mae5[:, vi]])
    both = both[np.isfinite(both)]
    vmin = np.nanpercentile(both, 2)
    vmax = np.nanpercentile(both, 98)

    for ri, (mae, lbl) in enumerate(zip([mae0, mae5], labels_mr)):
        ax = axes[ri, vi]
        vals = mae[:, vi]
        valid = np.isfinite(vals)
        # Simple DEM-free map (46_ doesn't have draw_dem)
        ax.scatter(stn.easting[valid], stn.northing[valid],
                   c=vals[valid], s=40, cmap="YlOrRd",
                   vmin=vmin, vmax=vmax,
                   edgecolors="0.4", linewidths=0.3, zorder=5)
        ax.set_aspect("equal")
        ax.grid(alpha=0.2)
        if ri == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if vi == 0:
            ax.set_ylabel(lbl, fontsize=9)

    # Colorbar for last column
    sm = plt.cm.ScalarMappable(cmap="YlOrRd",
                               norm=mcolors.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])

# ── Sync axes per variable column ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(2)]
    lo_y = min(ax.get_ylim()[0] for ax in col_axes)
    hi_y = max(ax.get_ylim()[1] for ax in col_axes)
    lo_x = min(ax.get_xlim()[0] for ax in col_axes)
    hi_x = max(ax.get_xlim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo_y, hi_y)
        ax.set_xlim(lo_x, hi_x)

# ── Sync wind component color scales ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
for ri in range(axes.shape[0]):
    sc_u = axes[ri, wu_i].collections[-1]
    sc_v = axes[ri, wv_i].collections[-1]
    vmin = min(sc_u.get_clim()[0], sc_v.get_clim()[0])
    vmax = max(sc_u.get_clim()[1], sc_v.get_clim()[1])
    sc_u.set_clim(vmin, vmax)
    sc_v.set_clim(vmin, vmax)

fig.suptitle("MAE Transformer visible-station MAE at +3 h — MR=0.0 vs MR=0.5",
             fontsize=12, y=1.01)
plt.tight_layout()
C.save_fig(fig, "46_vis_mae_map_mr_compare")
plt.show()
plt.close(fig)


In [ ]:
# ── Visible-station MAE vs lead time by TOD — MR=0.0 vs MR=0.5 ──────────────
TOD_LBL = ["00–06 UTC", "06–12 UTC", "12–18 UTC", "18–24 UTC"]

# Station sets: the MR=0.5 mask is redrawn every window (each station is
# visible in ~50% of windows), so EXT5's "vis" accumulators cover ALL 155
# stations (visible windows only) and EXT0 covers all 155 stations (all
# windows). Both curves are therefore over the same station set already;
# VIS_IDX = all stations. (An earlier version restricted to visible_bool[0],
# which is only the first window's partition — wrong for this dump.)
VIS_IDX = np.arange(N)

fig, axes = plt.subplots(4, NV, figsize=(17, 13), sharex=True)

for ti, tod_lbl in enumerate(TOD_LBL):
    for vi, v in enumerate(VARS):
        ax = axes[ti, vi]

        # MR=0.0: same stations as the MR=0.5 visible set
        mae0_tod = C.pool_stations(EXT0["tod_mod_sum"], EXT0["tod_mod_cnt"],
                                   station_idx=VIS_IDX)
        ax.plot(range(1, K), mae0_tod[ti, 1:, vi], "o-", ms=2.5, lw=1.2,
                color=C.MODELS["v27"][1], label="MAE Tr. MR=0.0")

        # MR=0.5: visible only (already restricted by the vis accumulator)
        mae5_tod = C.pool_stations(EXT5["tod_mod_vis_sum"],
                                   EXT5["tod_mod_vis_cnt"],
                                   station_idx=VIS_IDX)
        ax.plot(range(1, K), mae5_tod[ti, 1:, vi], "s--", ms=2.5, lw=1.2,
                color=C.MODELS["v27"][1], alpha=0.6, label="MAE Tr. MR=0.5 (vis)")

        ax.grid(alpha=0.3)
        if ti == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if ti == 3:
            ax.set_xticks(range(1, K, 2))
            ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
    axes[ti, 0].set_ylabel(f"{tod_lbl}\nMAE", fontsize=9)

# ── Sync y-axis per variable column across rows ──
for vi in range(NV):
    col_axes = [axes[ti, vi] for ti in range(4)]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)

axes[0, -1].legend(fontsize=6, loc="upper left")
fig.suptitle("MAE Transformer visible-station MAE vs lead time by TOD — "
             "MR=0.0 vs MR=0.5", y=1.01, fontsize=12)
plt.tight_layout()
C.save_fig(fig, "46_vis_mae_tod_mr_compare")
plt.show()
plt.close(fig)


In [ ]:
# ── Compute ΔMAE: local masking vs random control ───────────────────────────
#
# ΔMAE = pooled_MAE(HIGH) − pooled_MAE(LOW)
# where HIGH = (window, station) pairs with > 50% of R-km neighbours masked,
#       LOW  = pairs with ≤ 50% masked.
# Only pairs where the target is VISIBLE are included, isolating the effect
# of local context removal from self-masking.
#
# Random control: same statistic with randomly chosen pseudo-neighbours
# (same degree per station). Repeated N_TRIALS times.

THRESHOLD = 0.5
N_TRIALS = 50
rng = np.random.default_rng(42)

# ── Local ΔMAE ──────────────────────────────────────────────────────────────
dmae_local = {}  # (R, vi, tc_key) -> ΔMAE

for R in RADII_KM:
    lmc = local_masked_count[R]                          # (Mw, N)
    n_loc = N_LOCAL[R].astype(np.float32)                # (N,)
    has_nbr = n_loc > 0                                  # (N,)
    frac = np.where(has_nbr[None, :],
                    lmc / np.maximum(n_loc[None, :], 1), -1)  # (Mw, N)

    for vi in range(NV):
        vv = visible_bool & val5[:, :, vi]               # (Mw, N) visible AND valid
        vv_use = vv.copy()
        vv_use[:, ~has_nbr] = False                      # exclude isolated stations
        hi = vv_use & (frac > THRESHOLD)
        lo = vv_use & (frac <= THRESHOLD)
        e = err5[:, :, vi]

        for tc_key in ["all"] + list(TC_IDX):
            if tc_key == "all":
                tc_mask = np.ones(N, dtype=bool)
            else:
                tc_mask = np.zeros(N, dtype=bool)
                tc_mask[TC_IDX[tc_key]] = True

            tc_b = tc_mask[None, :].astype(e.dtype)  # (1, N)
            s_hi = (e * hi * tc_b).sum()
            c_hi = (hi & tc_mask[None, :]).sum()
            s_lo = (e * lo * tc_b).sum()
            c_lo = (lo & tc_mask[None, :]).sum()
            if c_hi > 50 and c_lo > 50:
                dmae_local[(R, vi, tc_key)] = s_hi / c_hi - s_lo / c_lo
            else:
                dmae_local[(R, vi, tc_key)] = np.nan

print("Local ΔMAE computed.")

# ── Random control ΔMAE ─────────────────────────────────────────────────────
dmae_rand = {}  # (R, vi) -> (mean, std)

for R in RADII_KM:
    n_loc = N_LOCAL[R]
    has_nbr = n_loc > 0

    # Pre-compute valid-visible masks per variable
    VV = {}
    for vi in range(NV):
        vv = (visible_bool & val5[:, :, vi]).copy()
        vv[:, ~has_nbr] = False
        VV[vi] = vv

    trial_vals = {vi: [] for vi in range(NV)}
    for trial in range(N_TRIALS):
        # Build random pseudo-adjacency with same degree per station
        pseudo_adj = np.zeros((N, N), dtype=np.float32)
        for i in range(N):
            ni = int(n_loc[i])
            if ni == 0:
                continue
            others = np.delete(np.arange(N), i)
            chosen = rng.choice(others, size=min(ni, len(others)), replace=False)
            pseudo_adj[i, chosen] = 1.0

        pseudo_lmc = masked_bool.astype(np.float32) @ pseudo_adj.T   # (Mw, N)
        pfrac = np.where(has_nbr[None, :],
                         pseudo_lmc / np.maximum(n_loc[None, :].astype(np.float32), 1),
                         -1)

        for vi in range(NV):
            vv = VV[vi]
            hi = vv & (pfrac > THRESHOLD)
            lo = vv & (pfrac <= THRESHOLD)
            e = err5[:, :, vi]
            s_hi, c_hi = (e * hi).sum(), hi.sum()
            s_lo, c_lo = (e * lo).sum(), lo.sum()
            if c_hi > 50 and c_lo > 50:
                trial_vals[vi].append(s_hi / c_hi - s_lo / c_lo)

    for vi in range(NV):
        if trial_vals[vi]:
            dmae_rand[(R, vi)] = (np.mean(trial_vals[vi]),
                                  np.std(trial_vals[vi]))
        else:
            dmae_rand[(R, vi)] = (np.nan, np.nan)

    print(f"  R={R} km: random control done ({N_TRIALS} trials)")

print("All ΔMAE computations complete.")

In [ ]:
# ── Main figure: ΔMAE vs masking radius ─────────────────────────────────────
# X-axis: radius (5, 10, 25, 50, 100 km)
# Y-axis: ΔMAE = MAE(high local masking) − MAE(low local masking)
# Lines: terrain classes only. Dashed: random control with ±1σ band.
fig, axes = plt.subplots(1, NV, figsize=(17, 4.0))
x = np.arange(len(RADII_KM))

for vi, (ax, v) in enumerate(zip(axes, VARS)):
    # By terrain class
    for tc in C.TCLASSES:
        y_tc = [dmae_local.get((R, vi, tc), np.nan) for R in RADII_KM]
        ax.plot(x, y_tc, "s--", lw=1.3, ms=5,
                color=TC_COLORS[tc], label=tc)

    # Random control with ±1σ band
    y_rand = np.array([dmae_rand.get((R, vi), (np.nan, 0))[0]
                       for R in RADII_KM])
    y_std  = np.array([dmae_rand.get((R, vi), (np.nan, 0))[1]
                       for R in RADII_KM])
    ax.plot(x, y_rand, "D:", lw=1.5, ms=5, color="0.5",
            label="Random control")
    ax.fill_between(x, y_rand - y_std, y_rand + y_std,
                    color="0.5", alpha=0.15)

    ax.axhline(0, ls=":", color="k", lw=0.6, alpha=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([str(R) for R in RADII_KM])
    ax.set_xlabel("Radius [km]", fontsize=9)
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
    ax.grid(alpha=.3)

axes[0].set_ylabel(f"ΔMAE at {LEAD[REF_KI]}\n(high − low local masking)")
axes[-1].legend(fontsize=5.5, loc="upper left")
fig.suptitle(f"Local-masking test, MAE Transformer at MR=0.5, {LEAD[REF_KI]}, visible stations only: MAE(>50% of R-km neighbours masked) − MAE(≤50%)\n"
             "by terrain class; grey = same statistic with random pseudo-neighbours (mean ±1σ, 50 draws)", y=1.06, fontsize=12)
plt.tight_layout()
C.save_fig(fig, "46_local_masking_dmae")
plt.show()
plt.close(fig)

### Extended masking analysis: all 10 radii

Same local-masking ΔMAE computation as above, extended to all 10
analysis radii. Fewer random-control trials (20) to keep runtime
manageable. The ΔMAE-vs-radius curve reveals the spatial scale at
which local context matters most.

## Dose-response: MAE vs local masking fraction

At R = 50 km, bin (window, station) pairs by the fraction of local
neighbours masked and plot the pooled MAE in each bin. A monotonic
increase confirms a causal dose-response relationship between local
context removal and forecast degradation.

In [ ]:
# ── Dose-response at R=50 km ────────────────────────────────────────────────
R_DOSE = 50
lmc = local_masked_count[R_DOSE]
n_loc = N_LOCAL[R_DOSE].astype(np.float32)
has_nbr = n_loc > 0
frac = np.where(has_nbr[None, :], lmc / np.maximum(n_loc[None, :], 1),
                np.nan)  # (Mw, N)

FRAC_BINS = np.linspace(0, 1, 6)  # 5 equal bins
BIN_LABELS = [f"{FRAC_BINS[i]:.1f}–{FRAC_BINS[i+1]:.1f}"
              for i in range(len(FRAC_BINS) - 1)]

fig, axes = plt.subplots(1, NV, figsize=(17, 3.5))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    vv = (visible_bool & val5[:, :, vi]).copy()
    vv[:, ~has_nbr] = False
    e = err5[:, :, vi]

    bin_mae = []
    for bi in range(len(FRAC_BINS) - 1):
        lo_b, hi_b = FRAC_BINS[bi], FRAC_BINS[bi + 1]
        if bi < len(FRAC_BINS) - 2:
            sel = vv & (frac >= lo_b) & (frac < hi_b)
        else:
            sel = vv & (frac >= lo_b) & (frac <= hi_b)
        if sel.sum() > 0:
            bin_mae.append((e * sel).sum() / sel.sum())
        else:
            bin_mae.append(np.nan)

    ax.bar(range(len(BIN_LABELS)), bin_mae, color="#3A7CA5",
           edgecolor="k", lw=0.5)
    ax.set_xticks(range(len(BIN_LABELS)))
    ax.set_xticklabels(BIN_LABELS, fontsize=7, rotation=30, ha="right")
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
    ax.grid(alpha=.3, axis="y")

axes[0].set_ylabel(f"MAE at {LEAD[REF_KI]}")
fig.suptitle(f"Dose-response: MAE vs fraction of local neighbours masked "
             f"(R={R_DOSE} km, v27 MR=0.5)", y=1.04)
plt.tight_layout()
C.save_fig(fig, "46_dose_response")
plt.show()
plt.close(fig)

## ΔMAE analysis: MR=0.0 vs MR=0.5 by network density

For each station, compute ΔMAE = MAE(MR=0.5) − MAE(MR=0.0) using
per-station aggregated statistics. All 155 stations have valid MAE at
both mask ratios (at MR=0.5, each station is visible in ~50% of
windows). Neighbour counts at R = 10, 25, 50, 75 km are used to
assess how ΔMAE scales with local network density.

A second view bins stations by neighbour-count categories and
compares absolute MAE at MR=0.0 and MR=0.5 side by side.

In [ ]:
# ── Per-station ΔMAE = MAE(MR=0.5) − MAE(MR=0.0) ──────────────────────────
AGG5 = C.load_agg("v27", "mr0.50")
a0 = AGG["v27"]

COMPARE_RADII = [10, 25, 50, 75]
NN_COMP = {R: (DIST <= R).sum(axis=1) for R in COMPARE_RADII}

mae_mr0 = np.where(a0["mod_all_cnt"][REF_KI] > 0,
                   a0["mod_all_sum_phys"][REF_KI] /
                   np.maximum(a0["mod_all_cnt"][REF_KI], 1),
                   np.nan)  # (N, NV)
mae_mr5 = np.where(AGG5["mod_all_cnt"][REF_KI] > 0,
                   AGG5["mod_all_sum_phys"][REF_KI] /
                   np.maximum(AGG5["mod_all_cnt"][REF_KI], 1),
                   np.nan)  # (N, NV)
delta_mae = mae_mr5 - mae_mr0  # (N, NV) — positive = masking hurts

# Summary table
for vi, v in enumerate(VARS):
    d = delta_mae[:, vi]
    valid = np.isfinite(d)
    print(f"{v}: median ΔMAE = {np.nanmedian(d):.4f}, "
          f"mean = {np.nanmean(d):.4f}, "
          f"positive: {(d[valid] > 0).sum()}/{valid.sum()} stations")

# ── ΔMAE vs neighbour count — scatter with regression ────────────────────
fig, axes = plt.subplots(len(COMPARE_RADII), NV,
                         figsize=(17, 3.5 * len(COMPARE_RADII)),
                         squeeze=False, constrained_layout=True)

for ri, R in enumerate(COMPARE_RADII):
    nn = NN_COMP[R]
    for vi, v in enumerate(VARS):
        ax = axes[ri, vi]
        d = delta_mae[:, vi]
        valid = np.isfinite(d)
        ax.scatter(nn[valid], d[valid], s=20, alpha=0.55,
                   c=nn[valid], cmap="viridis", edgecolors="0.4",
                   linewidths=0.3)
        ax.axhline(0, ls=":", color="k", lw=0.7, alpha=0.5)
        if valid.sum() > 5 and nn[valid].std() > 0:
            sl, ic, r, p, _ = linregress(nn[valid], d[valid])
            xfit = np.linspace(nn.min(), nn.max(), 50)
            ax.plot(xfit, sl * xfit + ic, "k--", lw=1.2, alpha=0.7)
            pstr = f"p={p:.1e}" if p < 0.001 else f"p={p:.3f}"
            ax.text(0.95, 0.95, f"r={r:.2f}\n{pstr}",
                    transform=ax.transAxes, fontsize=7, ha="right",
                    va="top", bbox=dict(boxstyle="round,pad=0.2",
                    fc="white", alpha=0.7))
        if ri == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        ax.grid(alpha=.3)
        if vi == 0:
            ax.set_ylabel(f"ΔMAE\n(R={R} km)", fontsize=9)
        if ri == len(COMPARE_RADII) - 1:
            ax.set_xlabel("# neighbours", fontsize=8)

fig.suptitle(f"ΔMAE (MR=0.5 − MR=0.0) vs neighbour count at "
             f"{LEAD[REF_KI]} — v27", fontsize=12)
# ── Sync y-axis per variable column across rows ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(len(COMPARE_RADII))]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)
C.save_fig(fig, "46_dmae_vs_nn_by_radius")
plt.show()
plt.close(fig)

# ── Summary: Pearson r(ΔMAE, NN) vs radius ──────────────────────────────
print("\nPearson r (ΔMAE vs NN count) by radius:")
for R in COMPARE_RADII:
    nn = NN_COMP[R]
    row = []
    for vi, v in enumerate(VARS):
        d = delta_mae[:, vi]
        valid = np.isfinite(d)
        r_val, p = pearsonr(nn[valid], d[valid])
        row.append(f"{v}: r={r_val:.3f} (p={p:.3f})")
    print(f"  R={R:>2d} km  " + "  ".join(row))


## Interaction analysis: spatial attention × network density

Two key interactions tested with existing predictions:

**SA × N** (at MR=0.0): spatial-attention gain = MAE(Blind) − MAE(Dense)
regressed against neighbour count N(R), distance-weighted availability
W_i = Σ exp(−d/λ), and nearest-station distance d_NN.

**MR × N** (v27 only): ΔMAE = MAE(MR=0.5) − MAE(MR=0.0) regressed
against W_i and d_NN as robustness checks alongside NN count.

The distance-weighted W_i (λ=25 km) captures spatial information
availability more continuously than discrete counts — two stations
with the same N(100) but different neighbour distances contribute
differently to W_i.

In [ ]:
# ── Spatial-attention gain per station at MR=0.0 ─────────────────────────────
a_blind = AGG["v32-blind"]
a_dense = AGG["v27"]  # v27 = dense spatial attention

cnt_b = a_blind["mod_all_cnt"][REF_KI]   # (N, NV)
cnt_d = a_dense["mod_all_cnt"][REF_KI]
mae_blind = np.where(cnt_b > 0, a_blind["mod_all_sum_phys"][REF_KI] /
                     np.maximum(cnt_b, 1), np.nan)
mae_dense = np.where(cnt_d > 0, a_dense["mod_all_sum_phys"][REF_KI] /
                     np.maximum(cnt_d, 1), np.nan)
sa_gain = mae_blind - mae_dense  # (N, NV) — positive = SA helps

for vi, v in enumerate(VARS):
    g = sa_gain[:, vi]
    valid = np.isfinite(g)
    print(f"{v}: median SA gain = {np.nanmedian(g):.4f}, "
          f"mean = {np.nanmean(g):.4f}, "
          f"positive: {(g[valid] > 0).sum()}/{valid.sum()}")


In [ ]:
# ── SA gain vs N(R), W_i, d_NN — scatter with regression ────────────────────
predictors = [
    ("N(10 km)", NN_COMP[10]),
    ("N(25 km)", NN_COMP[25]),
    ("N(50 km)", NN_COMP[50]),
    ("N(75 km)", NN_COMP[75]),
    (f"W_i (λ={LAMBDA_KM})", W_i),
    ("d_NN [km]", d_NN),
]

fig, axes = plt.subplots(len(predictors), NV,
                         figsize=(17, 3.0 * len(predictors)),
                         squeeze=False, constrained_layout=True)

sa_corr_table = []  # for summary

for pi, (pname, px) in enumerate(predictors):
    for vi, v in enumerate(VARS):
        ax = axes[pi, vi]
        g = sa_gain[:, vi]
        valid = np.isfinite(g) & np.isfinite(px)
        ax.scatter(px[valid], g[valid], s=15, alpha=0.5,
                   color="#D9663D", edgecolors="0.4", linewidths=0.3)
        ax.axhline(0, ls=":", color="k", lw=0.7, alpha=0.5)
        if valid.sum() > 5 and px[valid].std() > 0:
            sl, ic, r, p, _ = linregress(px[valid], g[valid])
            xfit = np.linspace(px[valid].min(), px[valid].max(), 50)
            ax.plot(xfit, sl * xfit + ic, "k--", lw=1.2, alpha=0.7)
            pstr = f"p={p:.1e}" if p < 0.001 else f"p={p:.3f}"
            ax.text(0.95, 0.95, f"r={r:.2f}\n{pstr}",
                    transform=ax.transAxes, fontsize=7, ha="right",
                    va="top", bbox=dict(boxstyle="round,pad=0.2",
                    fc="white", alpha=0.7))
            sa_corr_table.append({"predictor": pname, "variable": v,
                                  "r": r, "p": p, "n": int(valid.sum())})
        if pi == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        ax.grid(alpha=.3)
        if vi == 0:
            ax.set_ylabel(f"SA gain\n{pname}", fontsize=8)
        if pi == len(predictors) - 1:
            ax.set_xlabel(pname, fontsize=8)

fig.suptitle(f"Spatial-attention gain (MAE_blind − MAE_dense) vs "
             f"network density metrics at {LEAD[REF_KI]} (MR=0.0)",
             fontsize=12)
# ── Sync y-axis per variable column across rows ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(len(predictors))]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)
C.save_fig(fig, "46_sa_gain_vs_density")
plt.show()
plt.close(fig)


In [ ]:
# ── Summary: correlation table for all predictor × metric combos ────────────
print("=" * 80)
print("SA × N INTERACTION (MR=0.0): SA gain vs density predictors")
print("=" * 80)
df_sa = pd.DataFrame(sa_corr_table)
if len(df_sa) > 0:
    pivot = df_sa.pivot(index="predictor", columns="variable", values="r")
    # Reorder rows
    row_order = [p for p, _ in predictors if p in pivot.index]
    pivot = pivot.loc[row_order]
    print(pivot.round(3).to_string())
    C.save_table(pivot.reset_index(), "46_sa_gain_correlations")

print("\n" + "=" * 80)
print("MR × N INTERACTION (v27): ΔMAE vs density predictors")
print("=" * 80)
mr_corr = []
all_preds = [(f"N({R} km)", NN_COMP[R]) for R in COMPARE_RADII] + \
            [(f"W_i (λ={LAMBDA_KM})", W_i), ("d_NN [km]", d_NN)]
for pname, px in all_preds:
    for vi, v in enumerate(VARS):
        d = delta_mae[:, vi]
        valid = np.isfinite(d) & np.isfinite(px)
        if valid.sum() > 5 and px[valid].std() > 0:
            r_val, p = pearsonr(px[valid], d[valid])
            mr_corr.append({"predictor": pname, "variable": v,
                            "r": r_val, "p": p})
df_mr = pd.DataFrame(mr_corr)
if len(df_mr) > 0:
    pivot_mr = df_mr.pivot(index="predictor", columns="variable", values="r")
    row_order_mr = [p for p, _ in all_preds if p in pivot_mr.index]
    pivot_mr = pivot_mr.loc[row_order_mr]
    print(pivot_mr.round(3).to_string())
    C.save_table(pivot_mr.reset_index(), "46_dmae_correlations")

print("\nInterpretation guide:")
print("  SA gain vs N: positive r → SA helps more in dense areas (Story A)")
print("  SA gain vs d_NN: negative r → SA helps more for isolated stations (Story C)")
print("  ΔMAE vs N: positive r → denser stations lose more from masking")
print("  ΔMAE vs W_i: positive r → same, with distance-weighted measure")


### Minimum radius for 2 neighbours, by station

For each station, the smallest radius R such that at least 2 other
stations fall within it -- equivalently, the distance to its
**second**-nearest neighbour (the first-nearest alone isn't enough to
satisfy the ≥2-neighbours criterion used in the plot below).

In [ ]:
# ── Minimum radius for 2 neighbours, by station ─────────────────────────────
DIST_SORTED = np.sort(DIST, axis=1)          # (N, N), ascending, self excluded (inf)
min_r_2nn = DIST_SORTED[:, 1]                # distance to 2nd-nearest neighbour

df_min_r = pd.DataFrame({
    "station": stn.abbr.values,
    "min_radius_2nn_km": min_r_2nn,
}).sort_values("min_radius_2nn_km", ascending=False).reset_index(drop=True)

print(f"Minimum radius for \u22652 neighbours, by station "
      f"(median={np.median(min_r_2nn):.1f} km, "
      f"max={min_r_2nn.max():.1f} km):")
#display(df_min_r.style.format({"min_radius_2nn_km": "{:.1f}"}))


## MAE at +3h lead vs distance to k-th nearest neighbour — Dense vs Spatially Blind, by terrain class

Simplified to the two extremes of neighbour distance (1st and 5th
nearest) and the two MR=0.0 architecture endpoints: Dense (v31, full
spatial attention) vs Spatially Blind (v32-blind, no spatial
attention). Marker shape distinguishes the model, colour distinguishes
terrain class, and a Pearson-r regression line is fit per model. If
spatial attention is what drives the isolation effect, Dense should
show a real trend while Spatially Blind should not.

In [ ]:
# ── MAE at +3h lead vs distance to k-th nearest neighbour — Dense vs Spatially Blind, by terrain class ──
_, COL_DENSE, _ = C.MODELS["v31"]
_, COL_BLIND, _ = C.MODELS["v32-blind"]
MRK_DENSE, MRK_BLIND = "o", "^"

a_dense = AGG["v31"]
cnt_d = a_dense["mod_all_cnt"][REF_KI, :, :]
sum_d = a_dense["mod_all_sum_phys"][REF_KI, :, :]
mae_dense = np.where(cnt_d > 0, sum_d / np.maximum(cnt_d, 1), np.nan)

a_blind = AGG["v32-blind"]
cnt_b = a_blind["mod_all_cnt"][REF_KI, :, :]
sum_b = a_blind["mod_all_sum_phys"][REF_KI, :, :]
mae_blind = np.where(cnt_b > 0, sum_b / np.maximum(cnt_b, 1), np.nan)

K_NEIGHBOURS = [1, 5]
K_ORDINAL = {1: "1st", 5: "5th"}

tc_of_station = np.full(N, "", dtype=object)
for tc in C.TCLASSES:
    tc_of_station[TC_IDX[tc]] = tc


def _fit_line(ax, x, y, color):
    """Pearson-r linear fit, drawn + returned in the given colour."""
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 5 or x[valid].std() == 0:
        return None
    slope, intercept, r, p, _ = linregress(x[valid], y[valid])
    xfit = np.linspace(x[valid].min(), x[valid].max(), 50)
    ax.plot(xfit, slope * xfit + intercept, "--", lw=1.3, color=color, alpha=0.85)
    return r, p


fig, axes = plt.subplots(len(K_NEIGHBOURS), NV,
                         figsize=(17, 3.8 * len(K_NEIGHBOURS)),
                         squeeze=False, constrained_layout=True)
for ki, k in enumerate(K_NEIGHBOURS):
    dist_k = DIST_SORTED[:, k - 1]  # distance to k-th nearest neighbour
    for vi, v in enumerate(VARS):
        ax = axes[ki, vi]
        r_txt = []
        for model_label, mae_arr, mrk, mcol in [
            ("Dense", mae_dense, MRK_DENSE, COL_DENSE),
            ("Blind", mae_blind, MRK_BLIND, COL_BLIND),
        ]:
            y = mae_arr[:, vi]
            for tc in C.TCLASSES:
                sel = (tc_of_station == tc) & np.isfinite(y)
                ax.scatter(dist_k[sel], y[sel], s=28, alpha=0.75,
                           color=TC_COLORS[tc], marker=mrk,
                           edgecolors="k", linewidths=0.3)
            r = _fit_line(ax, dist_k, y, mcol)
            if r is not None:
                r_txt.append(f"{model_label} r={r[0]:.2f}")
        if r_txt:
            ax.text(0.97, 0.95, "\n".join(r_txt), transform=ax.transAxes,
                    fontsize=6.5, ha="right", va="top",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8))
        if ki == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if vi == 0:
            ax.set_ylabel(f"dist to {K_ORDINAL[k]} NN\nMAE", fontsize=8)
        if ki == len(K_NEIGHBOURS) - 1:
            ax.set_xlabel("Distance [km]", fontsize=8)
        ax.grid(alpha=.3)

model_handles = [
    plt.Line2D([], [], marker=MRK_DENSE, linestyle="", color="0.3", label="Dense"),
    plt.Line2D([], [], marker=MRK_BLIND, linestyle="", color="0.3", label="Spatially Blind"),
]
tc_handles = [plt.Line2D([], [], marker="o", linestyle="", color=TC_COLORS[tc], label=tc)
              for tc in C.TCLASSES]
fig.legend(handles=model_handles + tc_handles, loc="lower center", ncol=6,
          fontsize=7, bbox_to_anchor=(0.5, -0.03))
fig.suptitle(f"MAE at {LEAD[REF_KI]} vs distance to k-th nearest neighbour — "
             f"Dense vs Spatially Blind (MR=0), by terrain class", y=1.02)
# ── Sync y-axis per variable column across rows ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(len(K_NEIGHBOURS))]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)
C.save_fig(fig, "46_mae_vs_knn_dist_scatter")
plt.show()
plt.close(fig)


### Same scatter, faceted by time of day (3h bins, UTC)

Same station-level MAE-vs-isolation scatter as above, but split by
forecast-origin hour into eight 3-hour UTC bins (00–03, 03–06, ...,
21–24), to check whether the isolation effect is stable across the
diurnal cycle or concentrated at particular hours (e.g. overnight
cold-pool formation at valley-floor/isolated stations).

In [ ]:
# ── Same scatter, faceted by time of day (3h bins, UTC) ─────────────────────
# Self-contained: v27 visible/masked style constants and mask (the plot
# above this one has since been changed to Dense vs Spatially Blind and
# no longer defines these).
COL_MASKED, COL_VISIBLE = "#C4502A", "#1F5F6B"
MRK_MASKED, MRK_VISIBLE = "^", "o"
is_masked_v27 = masked_bool[0]  # FIRST window's mask only — the eval mask is redrawn every window (see load_mr05)

TH5 = TH5_full  # extracted in load_mr05 cell
t0h = TH5[:, 0]                            # forecast-origin hour, hours since epoch
hod = t0h % 24
# (no duplicate dump to delete), TH5

TOD3_LABELS = ["00–03", "03–06", "06–09", "09–12",
               "12–15", "15–18", "18–21", "21–24"]
tod3_idx = np.clip((hod // 3).astype(int), 0, len(TOD3_LABELS) - 1)

fig, axes = plt.subplots(len(TOD3_LABELS), NV,
                         figsize=(17, 3.0 * len(TOD3_LABELS)),
                         squeeze=False, constrained_layout=True)

for ti, tod_lbl in enumerate(TOD3_LABELS):
    win_sel = tod3_idx == ti
    for vi, v in enumerate(VARS):
        ax = axes[ti, vi]
        e  = err5[win_sel, :, vi]           # (Mw_ti, N)
        vv = val5[win_sel, :, vi]           # (Mw_ti, N) bool valid
        vis_w = visible_bool[win_sel, :]
        msk_w = masked_bool[win_sel, :]

        cnt_vis = (vv & vis_w).sum(axis=0)
        sum_vis = (e * (vv & vis_w)).sum(axis=0)
        mae_vis_t = np.where(cnt_vis > 0, sum_vis / np.maximum(cnt_vis, 1), np.nan)

        cnt_msk = (vv & msk_w).sum(axis=0)
        sum_msk = (e * (vv & msk_w)).sum(axis=0)
        mae_msk_t = np.where(cnt_msk > 0, sum_msk / np.maximum(cnt_msk, 1), np.nan)

        sel_vis = ~is_masked_v27 & np.isfinite(mae_vis_t)
        sel_msk = is_masked_v27 & np.isfinite(mae_msk_t)
        ax.scatter(min_r_2nn[sel_vis], mae_vis_t[sel_vis], s=18, alpha=0.6,
                   color=COL_VISIBLE, marker=MRK_VISIBLE,
                   label="visible" if (ti == 0 and vi == 0) else None)
        ax.scatter(min_r_2nn[sel_msk], mae_msk_t[sel_msk], s=18, alpha=0.6,
                   color=COL_MASKED, marker=MRK_MASKED,
                   label="masked" if (ti == 0 and vi == 0) else None)

        if ti == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=9)
        if vi == 0:
            ax.set_ylabel(f"{tod_lbl} UTC\nMAE", fontsize=8)
        if ti == len(TOD3_LABELS) - 1:
            ax.set_xlabel("Min radius for 2 neighbours [km]", fontsize=7)
        ax.grid(alpha=.3)

axes[0, 0].legend(fontsize=7, loc="upper left")
fig.suptitle(f"MAE at {LEAD[REF_KI]} vs min radius for 2 neighbours by time "
             f"of day — v27, MR=0.5 (visible vs masked)", y=1.005)
# ── Sync y-axis per variable column across rows ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(len(TOD3_LABELS))]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)
C.save_fig(fig, "46_mae_vs_isolation_scatter_tod3h")
plt.show()
plt.close(fig)


### MAE by exact neighbour count at R=10 km (0, 1, 3, 5 neighbours)

Complementary to the isolation bins above (which use the radius needed
to reach 2 neighbours): fix the radius at 10 km and ask whether MAE
degrades smoothly as the exact neighbour count steps up. Groups: 0, 1,
3, and 5 neighbours exactly, plus all other counts pooled as "other".

In [ ]:
# ── MAE by exact neighbour count at R=10 km (0, 1, 3, 5 neighbours) ─────────
EXACT_R = 10
n_at_r = N_LOCAL[EXACT_R]                     # (DIST <= 10 km).sum(axis=1), already computed
EXACT_COUNTS = [0, 1, 3, 5]
EXACT_LABELS = [f"{c} nbr" for c in EXACT_COUNTS] + ["other"]

exact_bin_idx = np.full(N, len(EXACT_COUNTS), dtype=int)  # default -> "other"
for ci, c in enumerate(EXACT_COUNTS):
    exact_bin_idx[n_at_r == c] = ci

print(f"Station counts by exact neighbour count at R={EXACT_R} km:")
for bi, lbl in enumerate(EXACT_LABELS):
    print(f"  {lbl:>8s}: {(exact_bin_idx == bi).sum():3d} stations")

fig, axes = plt.subplots(1, NV, figsize=(17, 3.8))
x_pos = np.arange(len(EXACT_LABELS))
w = 0.25
for mi, r in enumerate(["v27", "v32-blind", "lstm-baseline-v1"]):
    a = AGG[r]
    label, col, _ = C.MODELS[r]
    for vi, (ax, v) in enumerate(zip(axes, VARS)):
        cnt_a = a["mod_all_cnt"][REF_KI, :, vi]
        sum_a = a["mod_all_sum_phys"][REF_KI, :, vi]
        vals = []
        for bi in range(len(EXACT_LABELS)):
            sel = (exact_bin_idx == bi) & (cnt_a > 0)
            c = cnt_a[sel].sum()
            vals.append(sum_a[sel].sum() / c if c > 0 else np.nan)
        ax.bar(x_pos + mi * w, vals, width=w, color=col, label=label,
               edgecolor="k", linewidth=0.4)
        if mi == 0:
            ax.set_xticks(x_pos + w)
            ax.set_xticklabels(EXACT_LABELS, fontsize=7, rotation=20, ha="right")
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
            ax.grid(alpha=.3, axis="y")

axes[0].set_ylabel(f"MAE at {LEAD[REF_KI]}")
axes[-1].legend(fontsize=7)
fig.suptitle(f"MAE at {LEAD[REF_KI]} by exact neighbour count at R={EXACT_R} km",
             y=1.04)
plt.tight_layout()
C.save_fig(fig, "46_mae_by_exact_nn_count")
plt.show()
plt.close(fig)


## Supplementary: Nearest-station distance

Nearest-station distance as a supplementary isolation diagnostic -- a single scalar summary of how isolated a station is, complementing the neighbour-count radii above.

In [ ]:
# ── Supplementary: MAE vs nearest-station distance ───────────────────────────
fig, axes = plt.subplots(1, NV, figsize=(17, 3.5))
a = AGG["v27"]
nn_d = stn.nn_dist_km.values
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    cnt = a["mod_all_cnt"][REF_KI, :, vi]
    s = a["mod_all_sum_phys"][REF_KI, :, vi]
    mae = np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan)
    valid = ~np.isnan(mae)
    rp, pp = pearsonr(nn_d[valid], mae[valid])
    rs, ps = spearmanr(nn_d[valid], mae[valid])
    ax.scatter(nn_d[valid], mae[valid], s=15, alpha=0.6, color="#1F5F6B")
    ax.set_title(f"{v}  (ρ={rp:.2f}, ρₛ={rs:.2f})", fontsize=9)
    ax.set_xlabel("NN distance [km]", fontsize=8)
    ax.grid(alpha=.3)

axes[0].set_ylabel(f"MAE at {LEAD[REF_KI]}")
fig.suptitle("Supplementary: MAE Transformer MAE vs nearest-station distance (MR=0)",
             y=1.04)
plt.tight_layout()
C.save_fig(fig, "46_mae_vs_nn_dist")
plt.show()
plt.close(fig)

## Multi-radius neighbour-count analysis

For each station, count neighbours within 10 radii (10–95 km).
Scatter plots show per-station MAE and σₑ vs neighbour count with
regression lines. A compact Pearson r heatmap (radius × variable)
summarises the strength of correlation across all radii.

The masking experiment above uses the publication subset
(10, 25, 50, 100 km); the full 10-radius set is for exploratory
analysis and identifying the most informative radii.

In [ ]:
# ── Per-station MAE and σₑ for v27 at REF_KI ────────────────────────────────
a27 = AGG["v27"]
cnt_27 = a27["mod_all_cnt"][REF_KI, :, :]         # (N, NV)
sum_27 = a27["mod_all_sum_phys"][REF_KI, :, :]     # (N, NV)
sq_27  = a27["mod_all_sumsq_phys"][REF_KI, :, :]   # (N, NV)
sgn_27 = a27["mod_all_signed_phys"][REF_KI, :, :]  # (N, NV)

mae_27 = np.where(cnt_27 > 0, sum_27 / np.maximum(cnt_27, 1), np.nan)  # (N, NV)
n27 = np.maximum(cnt_27, 1)
sd_27  = np.sqrt(np.maximum(sq_27 / n27 - (sgn_27 / n27)**2, 0))       # (N, NV)
sd_27  = np.where(cnt_27 > 0, sd_27, np.nan)

# ── Pearson and Spearman correlations: (radius × variable) ──────────────────
corr_pearson_mae = np.full((len(NN_RADII), NV), np.nan)
corr_spearman_mae = np.full((len(NN_RADII), NV), np.nan)
corr_pearson_sd  = np.full((len(NN_RADII), NV), np.nan)
corr_spearman_sd = np.full((len(NN_RADII), NV), np.nan)
sample_sizes = np.zeros((len(NN_RADII), NV), dtype=int)

for ri, R in enumerate(NN_RADII):
    nn = NN_COUNT[R]
    for vi in range(NV):
        valid = np.isfinite(mae_27[:, vi]) & (nn > 0)
        sample_sizes[ri, vi] = valid.sum()
        if valid.sum() > 5:
            corr_pearson_mae[ri, vi], _ = pearsonr(nn[valid], mae_27[valid, vi])
            corr_spearman_mae[ri, vi], _ = spearmanr(nn[valid], mae_27[valid, vi])
            corr_pearson_sd[ri, vi], _ = pearsonr(nn[valid], sd_27[valid, vi])
            corr_spearman_sd[ri, vi], _ = spearmanr(nn[valid], sd_27[valid, vi])

print("Correlations computed.")
print(f"Sample sizes (min across radii×var): {sample_sizes.min()}")


### Correlation heatmaps: Pearson r (radius × variable)

Compact summary of how strongly neighbour count correlates with MAE
and σₑ across all 10 radii. Spearman ρ computed as robustness check
and printed below.

In [ ]:
# ── Pearson r heatmaps: MAE and σₑ vs NN count ──────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for ax, mat, title, cmap in [
    (ax1, corr_pearson_mae, "Pearson r: MAE vs NN count", "RdBu_r"),
    (ax2, corr_pearson_sd,  "Pearson r: σₑ vs NN count", "RdBu_r"),
]:
    vabs = max(np.nanmax(np.abs(mat)), 0.01)
    im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=-vabs, vmax=vabs)
    ax.set_xticks(range(NV))
    ax.set_xticklabels(VARS, fontsize=8, rotation=30, ha="right")
    ax.set_yticks(range(len(NN_RADII)))
    ax.set_yticklabels([f"{R} km" for R in NN_RADII], fontsize=8)
    ax.set_ylabel("Radius", fontsize=9)
    ax.set_title(title, fontsize=10)
    # Annotate each cell with r value
    for ri in range(len(NN_RADII)):
        for vi in range(NV):
            val = mat[ri, vi]
            if np.isfinite(val):
                color = "white" if abs(val) > vabs * 0.6 else "black"
                ax.text(vi, ri, f"{val:.2f}", ha="center", va="center",
                        fontsize=6.5, color=color)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)

fig.suptitle(f"Correlation between neighbour count and forecast error "
             f"metrics at {LEAD[REF_KI]} (v27, MR=0)", fontsize=12)
C.save_fig(fig, "46_corr_heatmap")
plt.show()
plt.close(fig)

# ── Spearman robustness check (printed, not plotted) ────────────────────────
print("\nSpearman ρ (MAE vs NN count):")
df_sp = pd.DataFrame(corr_spearman_mae, index=[f"{R} km" for R in NN_RADII],
                     columns=VARS)
print(df_sp.round(3).to_string())

print("\nSpearman ρ (σₑ vs NN count):")
df_sp_sd = pd.DataFrame(corr_spearman_sd, index=[f"{R} km" for R in NN_RADII],
                        columns=VARS)
print(df_sp_sd.round(3).to_string())

# Save tables
C.save_table(pd.DataFrame(corr_pearson_mae,
    index=[f"{R} km" for R in NN_RADII], columns=VARS
).reset_index().rename(columns={"index": "radius"}), "46_pearson_r_mae")


## Interpretation

The **local masking experiment** exploits the natural variation in
MR=0.5's random masking to estimate the causal effect of local spatial
context on forecast quality.

**ΔMAE vs radius**: at each radius R, the MAE increase when more than
half of a target station's R-km neighbours are masked (vs fewer than
half). A positive ΔMAE indicates that local context genuinely helps
the model. The terrain-class breakdown reveals whether stations in
complex terrain (valleys, ridges) are more sensitive to local masking.

**Random control**: replacing real spatial neighbours with randomly
chosen pseudo-neighbours eliminates the spatial structure. If the
random ΔMAE is near zero while the local ΔMAE is positive, the effect
is specifically spatial — the model relies on nearby stations, not
just on having more stations globally.

**Dose-response**: the monotonic increase in MAE with local masking
fraction at R=50 km confirms a causal relationship: more local masking
→ worse predictions, in a graded fashion.

**Aggregate masking impact**: the per-station ΔMAE (MR=0.5 − MR=0.0)
correlation with neighbour count reveals that stations in denser
network regions lose more absolute performance when masking is applied,
because they had more local context to lose.

## Visible-station MAE by neighbour-count group (R=50 km)

At R=50 km, split stations into four density groups (≤2, 3–5, 6–10,
>10 neighbours) and plot MAE vs lead time for MR=0.0 and MR=0.5.
If dense groups show a larger MR=0.5 penalty, masking hurts more
where the model has more spatial context to lose.

In [ ]:
# ── MAE vs lead time by neighbour-count group at R=50 km ─────────────────────
R_VIS = 50
nn50_vis = (DIST <= R_VIS).sum(axis=1)  # (N,)

NN_GROUPS = [
    ("≤2",   lambda nn: nn <= 2),
    ("3–5",  lambda nn: (nn >= 3) & (nn <= 5)),
    ("6–10", lambda nn: (nn >= 6) & (nn <= 10)),
    (">10",  lambda nn: nn > 10),
]
GROUP_COLORS = ["#BC4749", "#D4A373", "#5B8E7D", "#2B7A78"]

a0 = AGG["v27"]
a5 = AGG5  # reuse — already loaded in agg_impact cell

fig, axes = plt.subplots(2, NV, figsize=(17, 7),
                         squeeze=False, constrained_layout=True)

for mi, (agg, mr_label) in enumerate([(a0, "MR=0.0"), (a5, "MR=0.5")]):
    for vi, v in enumerate(VARS):
        ax = axes[mi, vi]
        for gi, (glabel, gfn) in enumerate(NN_GROUPS):
            mask = gfn(nn50_vis)
            n_stn = mask.sum()
            if n_stn == 0:
                continue
            # Pool MAE over stations in this group
            cnt = agg["mod_all_cnt"][:, mask, vi]   # (K, n_stn)
            s   = agg["mod_all_sum_phys"][:, mask, vi]
            mae_k = np.where(cnt.sum(axis=1) > 0,
                             s.sum(axis=1) / np.maximum(cnt.sum(axis=1), 1),
                             np.nan)  # (K,)
            ax.plot(range(1, K), mae_k[1:], "o-", ms=3, lw=1.3,
                    color=GROUP_COLORS[gi],
                    label=f"{glabel} (n={n_stn})" if mi == 0 else None)

        ax.set_xticks(range(1, K, 2))
        ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
        ax.grid(alpha=.3)
        if mi == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if vi == 0:
            ax.set_ylabel(f"MAE — {mr_label}", fontsize=9)

axes[0, -1].legend(fontsize=6.5, loc="upper left", title="NN within 50 km",
                   title_fontsize=7)
fig.suptitle("MAE Transformer MAE vs lead time by neighbour-count group (R=50 km)\n"
             "Top: MR=0.0 (full network) — Bottom: MR=0.5 (50% masked)",
             fontsize=12)
# ── Sync y-axis per variable column across rows ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(2)]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)
C.save_fig(fig, "46_visible_mr_compare_by_nn")
plt.show()
plt.close(fig)

# ── Print summary at REF_KI ─────────────────────────────────────────────────
print(f"\nMAE at {LEAD[REF_KI]} by group (R={R_VIS} km):")
group_hdr = "Group"; n_hdr = "n_stn"
print(f"{group_hdr:>8s}  {n_hdr:>5s}  ", end="")
for v in VARS:
    lbl0, lbl5, dlbl = v + "(0.0)", v + "(0.5)", "ΔMAE"
    print(f"  {lbl0:>12s}  {lbl5:>12s}  {dlbl:>8s}", end="")
print()
for glabel, gfn in NN_GROUPS:
    mask = gfn(nn50_vis)
    n_stn = mask.sum()
    print(f"{glabel:>8s}  {n_stn:>5d}  ", end="")
    for vi, v in enumerate(VARS):
        c0 = a0["mod_all_cnt"][REF_KI, mask, vi].sum()
        s0 = a0["mod_all_sum_phys"][REF_KI, mask, vi].sum()
        c5 = a5["mod_all_cnt"][REF_KI, mask, vi].sum()
        s5 = a5["mod_all_sum_phys"][REF_KI, mask, vi].sum()
        m0 = s0 / max(c0, 1)
        m5 = s5 / max(c5, 1)
        print(f"  {m0:>12.4f}  {m5:>12.4f}  {m5-m0:>+8.4f}", end="")
    print()


### MAE by minimum-neighbour threshold across radii

Stations with ≥2 neighbours within R form increasingly inclusive
clusters as R grows. This plot shows MAE vs lead time for each
cluster at R = 10, 15, 25, 50 km, with MR=0.0 (solid) and
MR=0.5 (dashed). The gap between solid and dashed reveals how
much masking hurts at each spatial scale.

In [ ]:
# ── MAE by "≥2 neighbours" cluster at R = 10, 15, 25, 50 km ────────────────
CLUSTER_RADII = [10, 15, 25, 50]
CLUSTER_COLORS = ["#BC4749", "#D4A373", "#5B8E7D", "#2B7A78"]
MIN_NN = 2

a0 = AGG["v27"]
a5 = AGG5  # reuse — already loaded in agg_impact cell

# Compute cluster membership
cluster_mask = {}
for R in CLUSTER_RADII:
    nn = (DIST <= R).sum(axis=1)
    cluster_mask[R] = nn >= MIN_NN

print("Stations with ≥2 neighbours by radius:")
for R in CLUSTER_RADII:
    print(f"  R={R:>2d} km: {cluster_mask[R].sum()} / {N} stations")

fig, axes = plt.subplots(1, NV, figsize=(17, 4.5),
                         constrained_layout=True)

for vi, v in enumerate(VARS):
    ax = axes[vi]
    for ci, R in enumerate(CLUSTER_RADII):
        mask = cluster_mask[R]
        n_stn = mask.sum()
        col = CLUSTER_COLORS[ci]

        # MR=0.0 — solid
        cnt0 = a0["mod_all_cnt"][:, mask, vi]
        sum0 = a0["mod_all_sum_phys"][:, mask, vi]
        mae0 = np.where(cnt0.sum(axis=1) > 0,
                        sum0.sum(axis=1) / np.maximum(cnt0.sum(axis=1), 1),
                        np.nan)

        # MR=0.5 — dashed
        cnt5 = a5["mod_all_cnt"][:, mask, vi]
        sum5 = a5["mod_all_sum_phys"][:, mask, vi]
        mae5 = np.where(cnt5.sum(axis=1) > 0,
                        sum5.sum(axis=1) / np.maximum(cnt5.sum(axis=1), 1),
                        np.nan)

        ax.plot(range(1, K), mae0[1:], "o-", ms=3, lw=1.5,
                color=col, label=f"R={R} km (n={n_stn}) — MR=0")
        ax.plot(range(1, K), mae5[1:], "s--", ms=3, lw=1.2,
                color=col, alpha=0.7,
                label=f"R={R} km — MR=0.5")

    ax.set_xticks(range(1, K, 2))
    ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
    ax.grid(alpha=.3)

axes[0].set_ylabel("MAE [phys]", fontsize=9)
axes[-1].legend(fontsize=5.5, loc="upper left", ncol=1)
fig.suptitle("MAE Transformer MAE vs lead time — stations with ≥2 neighbours\n"
             "Solid: MR=0.0 | Dashed: MR=0.5", fontsize=12)
C.save_fig(fig, "46_mae_by_nn_cluster")
plt.show()
plt.close(fig)


### Neighbour loss for visible stations (R=25 / 50 / 100 km)

Even though a visible station's own observations are available, its
*neighbours* may still be masked under MR=0.5. This counts, for each
visible station, how many of its R-km neighbours were masked by the
same fixed evaluation mask — the indirect information loss that could
explain part of the MR=0.0 vs MR=0.5 gap above via reduced cross-station
attention context, independent of the target station's own masking
status.

In [ ]:
# ── Neighbour loss for visible stations due to masking (R=25/50/100 km) ──────
NEIGHBOUR_RADII = [25, 50, 100]

means, stds, fracs = [], [], []
for R in NEIGHBOUR_RADII:
    lmc = local_masked_count[R][visible_bool]                       # (n_obs,)
    n_loc = np.broadcast_to(N_LOCAL[R][None, :], visible_bool.shape)[visible_bool]
    frac = lmc / np.maximum(n_loc, 1)
    means.append(lmc.mean()); stds.append(lmc.std()); fracs.append(frac.mean())

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(NEIGHBOUR_RADII))
ax.bar(x, means, yerr=stds, capsize=4, color="#3A7CA5",
       edgecolor="k", linewidth=0.5)
for xi, (m, s, f) in enumerate(zip(means, stds, fracs)):
    ax.text(xi, m + s + 0.3, f"{f:.0%} of neighbours", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([f"{R} km" for R in NEIGHBOUR_RADII])
ax.set_ylabel("# masked neighbours (mean \u00b1 std)")
ax.set_xlabel("Radius")
ax.set_title("Neighbour loss for visible stations under MR=0.5 masking (MAE Transformer)")
ax.grid(alpha=.3, axis="y")
plt.tight_layout()
C.save_fig(fig, "46_visible_neighbour_loss")
plt.show()
plt.close(fig)

print("Neighbour loss for visible stations (fixed evaluation mask):")
for R, m, s, f in zip(NEIGHBOUR_RADII, means, stds, fracs):
    print(f"  R={R:>3d} km: mean masked neighbours = {m:.2f} \u00b1 {s:.2f}  "
          f"({f:.1%} of local neighbours)")
